# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Final Sample Construction

---
## Foreword

In this notebook, my goal is to build the final main sample for the models developed in my thesis.


## 1. Libraries & Data

I first load the relevant libraries and data.

In [42]:
# libs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [50]:
# data
main = pd.read_csv("../../../data/processed/main_with_svi.csv")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_4467/1285260102.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../../data/processed/main_with_svi.csv")


## 2. Missing Values & EDA

In this section, the goal is to identify the missing values that will be an issue for my analysis. The main constraint I will face is the serious missingness in Google Trends search volume index (SVI).

In [51]:
# parse dates
main['date'] = pd.to_datetime(main['date'])

# rename TSSVI to svi
main.rename(columns={'TSSVI': 'svi'}, inplace=True)

### 2.1. Missing Google Trends Data

I study the missingness of Google Trends data.

In [52]:
# number of unique tickers in the google svi dataset
svi = pd.read_csv("../../../data/interim/google_trends/google_trends_svi.csv")

# unique tickers in svi dataset
unique_tickers_svi = svi['ticker'].unique()
print(f"Number of unique tickers in SVI dataset: {len(unique_tickers_svi)}")

# unique tickers in main dataset
unique_tickers_main = main['ticker'].unique()
print(f"Number of unique tickers in main dataset: {len(unique_tickers_main)}")

# intersection of tickers in main and svi datasets
intersection_tickers = set(unique_tickers_main).intersection(set(unique_tickers_svi))
print(f"Number of tickers in intersection: {len(intersection_tickers)}")

Number of unique tickers in SVI dataset: 3034
Number of unique tickers in main dataset: 9007
Number of tickers in intersection: 2179


This means that in the attention sample I will have 2,179 tickers.

#### herding events

I also study the influence on the number of herding events.

In [53]:
# sort values by ticker and date
rh_herd = main[main["ticker"].isin(intersection_tickers)].copy()  # create a copy to avoid SettingWithCopyWarning
rh_herd = rh_herd.sort_values(["ticker", "date"]).copy()

# userratio = users_close(t) / users_close(t-1)
rh_herd["users_close_lag1"] = rh_herd.groupby("ticker")["users_close"].shift(1)
rh_herd["userratio"] = rh_herd["users_close"] / rh_herd["users_close_lag1"]

# flag np.inf and -np.inf values in userratio as NaN (can happen when users_close_lag1 is zero)
rh_herd["userratio"].replace([np.inf, -np.inf], np.nan, inplace=True)

# shortlist the possible herding events based on the criteria of Barbet et al. (2022)
rh_herd["shortlist_herd"] = (
    (rh_herd["userratio"] > 1)
    & (rh_herd["users_close_lag1"] >= 100)
    & rh_herd["userratio"].notna()
)

# Identify the top 0.5% of shortlisted stocks based on userratio for each day
# Only shortlisted stocks can be flagged as herding events.
rh_herd["rh_herd"] = False
rh_herd.loc[rh_herd["shortlist_herd"], "rh_herd"] = (
    rh_herd.loc[rh_herd["shortlist_herd"]]
    .groupby("date")["userratio"]
    .transform(lambda x: x >= x.quantile(0.995))
)

# Report on the number of herding events identified, number of
# unique tickers involved
num_herding_events = rh_herd["rh_herd"].sum()
num_unique_tickers = rh_herd.loc[rh_herd["rh_herd"], "ticker"].nunique()
print(f"Number of herding events identified: {num_herding_events}")
print(f"Number of unique tickers involved: {num_unique_tickers}")

Number of herding events identified: 1997
Number of unique tickers involved: 864


/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_4467/4283351143.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  rh_herd["userratio"].replace([np.inf, -np.inf], np.nan, inplace=True)


This means that I still have a meaningful number of herding events.

#### summary statistics

I compute the summary statistics to compare it to my previous sample.

In [54]:
# restrict main to the tickers in the intersection of main and svi datasets
main = main[main["ticker"].isin(intersection_tickers)].copy()

In [55]:
# users_close_l1: lagged users_close by 1 day
main["users_close_l1"] = main.groupby("ticker")["users_close"].shift(1)

# userchg: change in users from day t-1 to day t
main["userchg"] = main["users_close"] - main["users_close_l1"]

# userratio: users_close(t) / users_close(t-1)
main["userratio"] = main["users_close"] / main["users_close_l1"]

# replace inf and -inf in userratio with NaN
main["userratio"] = main["userratio"].replace([np.inf, -np.inf], np.nan)

# make sure that I take the absolute value of the price
main["prc"] = main["prc"].abs()

# size is defined as prc * shrout
main["size"] = main["prc"] * main["shrout"] / 1e6  # in millions

# build 'daily_buys'
main["daily_buys"] = main["buy_num_trades_retail"]

# build 'daily_sells'
main["daily_sells"] = main["sell_num_trades_retail"]

# build 'net_buys'
main["net_buys"] = main["daily_buys"] - main["daily_sells"]

# build 'taq_retimb'
main['taq_retimb'] = main['net_buys'] / (main['daily_buys'] + main['daily_sells'])

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
main["has_users"] = (main["users_close"] > 0).astype(int)

In [56]:
# summary stats Panel A
summary_stats_panel_a = main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb", "svi"]].describe().T
print(summary_stats_panel_a)

                 count         mean           std           min        25%  \
users_close  1029053.0  3844.786871  24275.548388      0.000000  97.000000   
users_last   1031704.0  3842.791364  24270.693324      0.000000  97.000000   
userchg      1020615.0    17.073472    334.962877 -17015.000000  -2.000000   
userratio    1017964.0     1.006915      0.442229      0.000000   0.995833   
prc          1186461.0    65.607142    148.757162      0.030000  15.420000   
size         1186461.0    13.905867     54.356667      0.000092   0.633956   
ret          1186461.0     0.000292      0.037056     -1.000000  -0.011407   
daily_buys   1178709.0   330.359598   1572.228190      0.000000  23.000000   
daily_sells  1178709.0   303.511857   1235.031978      0.000000  23.000000   
net_buys     1178709.0    26.847741    521.833182 -24007.000000 -12.000000   
taq_retimb   1178709.0     0.001005      0.239256     -1.000000  -0.102703   
svi          1145916.0    19.238877     29.838709      0.000000 

## 3. Building the Final Sample

For the final sample, I will keep only the Russell 3000 stocks used in the google trends data set. This means that the scope of my analysis will be more restricted than Barber et al. (2022).

In [57]:
# read data
df_final = pd.read_csv("../../../data/processed/main_with_svi.csv")

# parse dates
df_final['date'] = pd.to_datetime(df_final['date'])

# rename TSSVI to svi
df_final.rename(columns={'TSSVI': 'svi'}, inplace=True)

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_4467/571122779.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv("../../../data/processed/main_with_svi.csv")


In [62]:
# new main
new_main = df_final.loc[df_final["ticker"].isin(intersection_tickers)].copy()

# build relevant variables for new_main
new_main["users_close_l1"] = new_main.groupby("ticker")["users_close"].shift(1)

# userchg: change in users from day t-1 to day t
new_main["userchg"] = new_main["users_close"] - new_main["users_close_l1"]

# userratio: users_close(t) / users_close(t-1)
new_main["userratio"] = new_main["users_close"] / new_main["users_close_l1"]

# replace inf and -inf in userratio with NaN
new_main["userratio"] = new_main["userratio"].replace([np.inf, -np.inf], np.nan)

# make sure that I take the absolute value of the price
new_main["prc"] = new_main["prc"].abs()

# size is defined as prc * shrout
new_main["size"] = new_main["prc"] * new_main["shrout"] / 1e6  # in millions

# build 'daily_buys'
new_main["daily_buys"] = new_main["buy_num_trades_retail"]

# build 'daily_sells'
new_main["daily_sells"] = new_main["sell_num_trades_retail"]

# build 'net_buys'
new_main["net_buys"] = new_main["daily_buys"] - new_main["daily_sells"]

# build 'taq_retimb'
new_main['taq_retimb'] = new_main['net_buys'] / (new_main['daily_buys'] + new_main['daily_sells'])

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
new_main["has_users"] = (new_main["users_close"] > 0).astype(int)

# build abnormal SVI
def make_ab_svi(s):
    lags = pd.concat([s.shift(k) for k in range(2, 22)], axis=1)
    mu = lags.mean(axis=1)
    ratio = s.shift(1) / mu
    ratio[mu == 0] = np.nan
    return np.log1p(np.abs(ratio))

new_main["asvi"] = new_main.groupby("ticker")["svi"].transform(make_ab_svi)

Once I computed the relevant variables, I compute summary statistics.

In [63]:
# summary stats Panel A
summary_stats_panel_a = new_main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb", "svi", "asvi"]].describe().T
print(summary_stats_panel_a)

                 count         mean           std           min        25%  \
users_close  1029053.0  3844.786871  24275.548388      0.000000  97.000000   
users_last   1031704.0  3842.791364  24270.693324      0.000000  97.000000   
userchg      1020615.0    17.073472    334.962877 -17015.000000  -2.000000   
userratio    1017964.0     1.006915      0.442229      0.000000   0.995833   
prc          1186461.0    65.607142    148.757162      0.030000  15.420000   
size         1186461.0    13.905867     54.356667      0.000092   0.633956   
ret          1186461.0     0.000292      0.037056     -1.000000  -0.011407   
daily_buys   1178709.0   330.359598   1572.228190      0.000000  23.000000   
daily_sells  1178709.0   303.511857   1235.031978      0.000000  23.000000   
net_buys     1178709.0    26.847741    521.833182 -24007.000000 -12.000000   
taq_retimb   1178709.0     0.001005      0.239256     -1.000000  -0.102703   
svi          1145916.0    19.238877     29.838709      0.000000 

In [66]:
# Panel B: Daily Observations (Summed Variable Averaged across Days)
# keep only observations where has_users == 1
new_main_agg = new_main[new_main["has_users"] == 1].copy()

# group by date and sum the relevant variables: [users_close, users_last, userchg, daily_buys, daily_sells, net_buys]
daily_summary = new_main_agg.groupby("date").agg({
    "users_close": "sum",
    "users_last": "sum",
    "userchg": "sum",
    "daily_buys": "sum",
    "daily_sells": "sum",
    "net_buys": "sum",
    "has_users": "sum"
}).reset_index()

# summary stats Panel B
summary_stats_panel_b = daily_summary[["users_close", "users_last", "userchg", "daily_buys", "daily_sells", "net_buys", "has_users"]].describe().T
print(summary_stats_panel_b)

             count          mean           std        min        25%  \
users_close  543.0  7.286353e+06  4.856337e+06  2662756.0  4326191.5   
users_last   543.0  7.291845e+06  4.861880e+06  2663632.0  4326601.5   
userchg      543.0  3.209680e+04  6.380515e+04   -64808.0     4244.5   
daily_buys   543.0  6.398335e+05  3.594367e+05   215515.0   422414.0   
daily_sells  543.0  5.859621e+05  2.804619e+05   203500.0   415239.5   
net_buys     543.0  5.387139e+04  9.193979e+04   -55156.0     3047.5   
has_users    543.0  1.890203e+03  3.779899e+01     1116.0     1874.0   

                   50%        75%         max  
users_close  5474933.0  6577280.0  20340538.0  
users_last   5475312.0  6577989.0  20349987.0  
userchg         8894.0    21599.0    467921.0  
daily_buys    472933.0   660797.5   2128926.0  
daily_sells   461517.0   612002.0   1732292.0  
net_buys       17387.0    51688.5    579203.0  
has_users       1890.0     1906.0      1928.0  


I write `new_main` to csv

In [67]:
# write new main to csv
new_main.to_csv("../../../data/processed/final_sample.csv", index=False)

## 4. Summary Statistics Final Sample

In [69]:
# load the data 
df = pd.read_csv("../../../data/processed/final_sample.csv")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_4467/3467921232.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../../data/processed/final_sample.csv")


In [70]:
# Panel A: Stock-Day Observations (Averaged across Stocks)
summary_stats_panel_a = df[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret", "daily_buys", "daily_sells", "net_buys", "taq_retimb", "svi", "asvi"]].describe().T
print(summary_stats_panel_a)

                 count         mean           std           min        25%  \
users_close  1029053.0  3844.786871  24275.548388      0.000000  97.000000   
users_last   1031704.0  3842.791364  24270.693324      0.000000  97.000000   
userchg      1020615.0    17.073472    334.962877 -17015.000000  -2.000000   
userratio    1017964.0     1.006915      0.442229      0.000000   0.995833   
prc          1186461.0    65.607142    148.757162      0.030000  15.420000   
size         1186461.0    13.905867     54.356667      0.000092   0.633956   
ret          1186461.0     0.000292      0.037056     -1.000000  -0.011407   
daily_buys   1178709.0   330.359598   1572.228190      0.000000  23.000000   
daily_sells  1178709.0   303.511857   1235.031978      0.000000  23.000000   
net_buys     1178709.0    26.847741    521.833182 -24007.000000 -12.000000   
taq_retimb   1178709.0     0.001005      0.239256     -1.000000  -0.102703   
svi          1145916.0    19.238877     29.838709      0.000000 

In [71]:
# Panel B: Daily Observations (Summed Variable Averaged across Days)
summary_stats_panel_b = daily_summary[["users_close", "users_last", "userchg", "daily_buys", "daily_sells", "net_buys", "has_users"]].describe().T
print(summary_stats_panel_b)

             count          mean           std        min        25%  \
users_close  543.0  7.286353e+06  4.856337e+06  2662756.0  4326191.5   
users_last   543.0  7.291845e+06  4.861880e+06  2663632.0  4326601.5   
userchg      543.0  3.209680e+04  6.380515e+04   -64808.0     4244.5   
daily_buys   543.0  6.398335e+05  3.594367e+05   215515.0   422414.0   
daily_sells  543.0  5.859621e+05  2.804619e+05   203500.0   415239.5   
net_buys     543.0  5.387139e+04  9.193979e+04   -55156.0     3047.5   
has_users    543.0  1.890203e+03  3.779899e+01     1116.0     1874.0   

                   50%        75%         max  
users_close  5474933.0  6577280.0  20340538.0  
users_last   5475312.0  6577989.0  20349987.0  
userchg         8894.0    21599.0    467921.0  
daily_buys    472933.0   660797.5   2128926.0  
daily_sells   461517.0   612002.0   1732292.0  
net_buys       17387.0    51688.5    579203.0  
has_users       1890.0     1906.0      1928.0  


In [76]:
# number of positive svi measures
num_positive_svi = (df["svi"] > 0).sum()
num_above_50 = (df["svi"] > 50).sum()
num_above_75 = (df["svi"] == 100).sum()
print(f"Percentage of positive SVI measures: {num_positive_svi / len(df) * 100:.2f}%")
print(f"Percentage of SVI measures above 50: {num_above_50 / len(df) * 100:.2f}%")
print(f"Percentage of SVI measures equal to 100: {num_above_75 / len(df) * 100:.2f}%")

Percentage of positive SVI measures: 34.14%
Percentage of SVI measures above 50: 16.93%
Percentage of SVI measures equal to 100: 3.14%


In [77]:
# control variables from RavenPack
control_vars = ["ess", "css", "anl_chg", "num_news", "num_news_relevant"]

# percentage of observations with non-missing values for each control variable
for var in control_vars:
    non_missing_percentage = df[var].notna().mean() * 100
    print(f"Percentage of non-missing values for {var}: {non_missing_percentage:.2f}%")

Percentage of non-missing values for ess: 41.11%
Percentage of non-missing values for css: 57.10%
Percentage of non-missing values for anl_chg: 57.10%
Percentage of non-missing values for num_news: 57.10%
Percentage of non-missing values for num_news_relevant: 47.28%
